# Train Faster R-CNN, EfficientDet, RT-DETR

Use two Roboflow exports:

- COCO JSON for Faster R-CNN and EfficientDet.
- YOLOv8/YOLOv11 for RT-DETR through Ultralytics.


## Mount Google Drive


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


## Install Dependencies


In [ ]:

!pip install -q ultralytics effdet pycocotools


## Configuration


In [ ]:

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pycocotools.coco import COCO

# Roboflow COCO JSON export for Faster R-CNN and EfficientDet.
COCO_ROOT = Path('/content/shrimp-coco')
COCO_TRAIN_JSON = COCO_ROOT / 'train' / '_annotations.coco.json'
COCO_VAL_JSON = COCO_ROOT / 'valid' / '_annotations.coco.json'
COCO_TEST_JSON = COCO_ROOT / 'test' / '_annotations.coco.json'

# Roboflow YOLOv8/YOLOv11 export for RT-DETR through Ultralytics.
YOLO_DATA_YAML = Path('/content/shrimp-yolo/data.yaml')

OUTPUT_ROOT = Path('/content/drive/MyDrive/shrimp/runs_detection_models')

IMG_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 50
SEEDS = [0, 1, 2]
NUM_WORKERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE)
print('coco root:', COCO_ROOT)
print('yolo data yaml:', YOLO_DATA_YAML)
print('output root:', OUTPUT_ROOT)


## COCO Dataset Utilities


In [ ]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class CocoDetectionDataset(Dataset):
    def __init__(self, annotation_file: Path, img_size: int = 512):
        self.annotation_file = Path(annotation_file)
        self.image_dir = self.annotation_file.parent
        self.img_size = img_size
        self.coco = COCO(str(self.annotation_file))
        self.image_ids = sorted(self.coco.getImgIds())
        self.cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_label = {cat_id: idx + 1 for idx, cat_id in enumerate(self.cat_ids)}
        self.label_to_cat_id = {label: cat_id for cat_id, label in self.cat_id_to_label.items()}
        self.class_names = [self.coco.cats[cat_id]['name'] for cat_id in self.cat_ids]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.coco.loadImgs(image_id)[0]
        image_path = self.image_dir / image_info['file_name']
        image = Image.open(image_path).convert('RGB')
        orig_w, orig_h = image.size
        image = image.resize((self.img_size, self.img_size))
        image_tensor = F.to_tensor(image)

        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h

        ann_ids = self.coco.getAnnIds(imgIds=image_id, iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []
        areas = []
        iscrowd = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                continue
            x1 = max(0.0, x * scale_x)
            y1 = max(0.0, y * scale_y)
            x2 = min(float(self.img_size - 1), (x + w) * scale_x)
            y2 = min(float(self.img_size - 1), (y + h) * scale_y)
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_label[ann['category_id']])
            areas.append((x2 - x1) * (y2 - y1))
            iscrowd.append(int(ann.get('iscrowd', 0)))

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([image_id]),
            'area': torch.tensor(areas, dtype=torch.float32),
            'iscrowd': torch.tensor(iscrowd, dtype=torch.int64),
        }
        return image_tensor, target


def detection_collate(batch):
    return tuple(zip(*batch))


train_ds = CocoDetectionDataset(COCO_TRAIN_JSON, IMG_SIZE)
val_ds = CocoDetectionDataset(COCO_VAL_JSON, IMG_SIZE)
test_ds = CocoDetectionDataset(COCO_TEST_JSON, IMG_SIZE)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print('classes:', CLASS_NAMES)
print('train images:', len(train_ds), 'val images:', len(val_ds), 'test images:', len(test_ds))


## Faster R-CNN Training


In [ ]:

def build_faster_rcnn(num_classes: int):
    # num_classes includes background class.
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def train_faster_rcnn(seed: int):
    set_seed(seed)
    run_name = f'fasterrcnn_resnet50_fpn_seed{seed}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=detection_collate,
    )

    model = build_faster_rcnn(NUM_CLASSES + 1).to(DEVICE)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.005,
        momentum=0.9,
        weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

    history = []
    best_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        mean_loss = float(np.mean(losses))
        history.append({'epoch': epoch, 'train_loss': mean_loss})
        print(f'[{run_name}] epoch {epoch:03d}/{EPOCHS} train_loss={mean_loss:.4f}')

        if mean_loss < best_loss:
            best_loss = mean_loss
            torch.save({
                'model_state': model.state_dict(),
                'class_names': CLASS_NAMES,
                'img_size': IMG_SIZE,
                'epoch': epoch,
                'train_loss': mean_loss,
            }, run_dir / 'best.pt')

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    torch.save(model.state_dict(), run_dir / 'last_state_dict.pt')
    return run_dir


for seed in SEEDS:
    train_faster_rcnn(seed)


## EfficientDet Training


In [ ]:

from effdet import create_model


class EfficientDetCocoDataset(CocoDetectionDataset):
    def __getitem__(self, idx):
        image, target = super().__getitem__(idx)
        boxes_xyxy = target['boxes']
        if len(boxes_xyxy):
            # effdet expects yxyx boxes and 0-based class ids.
            boxes_yxyx = boxes_xyxy[:, [1, 0, 3, 2]]
            labels = target['labels'] - 1
        else:
            boxes_yxyx = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        return image, {'bbox': boxes_yxyx, 'cls': labels}


def efficientdet_collate(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)

    max_boxes = max(t['bbox'].shape[0] for t in targets)
    if max_boxes == 0:
        max_boxes = 1

    bbox = torch.zeros((len(targets), max_boxes, 4), dtype=torch.float32)
    cls = torch.full((len(targets), max_boxes), -1, dtype=torch.int64)
    img_scale = torch.ones((len(targets),), dtype=torch.float32)
    img_size = torch.full((len(targets), 2), IMG_SIZE, dtype=torch.float32)

    for i, t in enumerate(targets):
        n = t['bbox'].shape[0]
        if n:
            bbox[i, :n] = t['bbox']
            cls[i, :n] = t['cls']

    return images, {
        'bbox': bbox,
        'cls': cls,
        'img_scale': img_scale,
        'img_size': img_size,
    }


eff_train_ds = EfficientDetCocoDataset(COCO_TRAIN_JSON, IMG_SIZE)


def build_efficientdet(num_classes: int):
    return create_model(
        'tf_efficientdet_d0',
        bench_task='train',
        num_classes=num_classes,
        pretrained=True,
        image_size=(IMG_SIZE, IMG_SIZE),
    )


def train_efficientdet(seed: int):
    set_seed(seed)
    run_name = f'efficientdet_d0_seed{seed}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = DataLoader(
        eff_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=efficientdet_collate,
    )

    model = build_efficientdet(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = []
    best_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for images, targets in train_loader:
            images = images.to(DEVICE)
            targets = {k: v.to(DEVICE) for k, v in targets.items()}

            loss_dict = model(images, targets)
            loss = loss_dict['loss'] if isinstance(loss_dict, dict) else loss_dict

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        mean_loss = float(np.mean(losses))
        history.append({'epoch': epoch, 'train_loss': mean_loss})
        print(f'[{run_name}] epoch {epoch:03d}/{EPOCHS} train_loss={mean_loss:.4f}')

        if mean_loss < best_loss:
            best_loss = mean_loss
            torch.save({
                'model_state': model.state_dict(),
                'class_names': CLASS_NAMES,
                'img_size': IMG_SIZE,
                'epoch': epoch,
                'train_loss': mean_loss,
            }, run_dir / 'best.pt')

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    torch.save(model.state_dict(), run_dir / 'last_state_dict.pt')
    return run_dir


for seed in SEEDS:
    train_efficientdet(seed)


## RT-DETR Training


In [ ]:

from ultralytics import RTDETR


def train_rtdetr(seed: int):
    set_seed(seed)
    run_name = f'rtdetr_l_seed{seed}'
    print(f'========== TRAIN RT-DETR | seed={seed} | run={run_name} ==========')

    model = RTDETR('rtdetr-l.pt')
    model.train(
        data=str(YOLO_DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=20,
        project=str(OUTPUT_ROOT),
        name=run_name,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=seed,
        deterministic=True,
    )


for seed in SEEDS:
    train_rtdetr(seed)


## RT-DETR Test Evaluation


In [ ]:

from ultralytics import RTDETR

rows = []
for seed in SEEDS:
    run_name = f'rtdetr_l_seed{seed}'
    model_path = OUTPUT_ROOT / run_name / 'weights' / 'best.pt'
    if not model_path.exists():
        print('[WARN] missing:', model_path)
        continue

    model = RTDETR(str(model_path))
    metrics = model.val(
        data=str(YOLO_DATA_YAML),
        split='test',
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0 if DEVICE == 'cuda' else 'cpu',
        verbose=False,
    )
    rows.append({
        'model': 'RT-DETR-L',
        'seed': seed,
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'preprocess_ms': float(metrics.speed.get('preprocess', 0.0)),
        'inference_ms': float(metrics.speed.get('inference', 0.0)),
        'postprocess_ms': float(metrics.speed.get('postprocess', 0.0)),
    })

rtdetr_df = pd.DataFrame(rows)
rtdetr_df.to_csv(OUTPUT_ROOT / 'rtdetr_test_summary.csv', index=False)
rtdetr_df


## Notes

- Put Roboflow COCO JSON export at `/content/shrimp-coco` for Faster R-CNN and EfficientDet.
- Put Roboflow YOLOv8/YOLOv11 export at `/content/shrimp-yolo` for RT-DETR.
- Faster R-CNN checkpoint: `runs_detection_models/fasterrcnn_resnet50_fpn_seed*/best.pt`.
- EfficientDet checkpoint: `runs_detection_models/efficientdet_d0_seed*/best.pt`.
- RT-DETR checkpoint: `runs_detection_models/rtdetr_l_seed*/weights/best.pt`.
- If Colab runs out of VRAM, reduce `BATCH_SIZE` to `2` or `1`.
